# Imports

In [ ]:
from __future__ import annotations

import warnings
import json

import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import HyperParams
from src.data import load_data
from src.model_base import MiniBERT
from src.model_ode import MiniBERTNeuralODE
from src.training import train
from src.evaluate import evaluate

warnings.filterwarnings("ignore")

# Inits

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)

In [ ]:
hp = HyperParams()
print("Using hyperparameters:", json.dumps(hp.__dict__, indent=2))

# Data

In [ ]:
data = load_data(hp)
labels = data["label_names"]
data["tokenizer"].save("artifacts/tokenizer.json")

# BERT

In [ ]:
baseline = MiniBERT(hp=hp).to(device)

## Train

In [ ]:
train(
    model=baseline, 
    data=data,
    device=device,
    hp=hp,
    model_path="artifacts/baseline.pt",
    history_path="artifacts/baseline.json"
)

## Eval

In [ ]:
_, _, report, cm = evaluate(baseline, data, device)
print(report)

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=labels,
    yticklabels=labels,
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# ODE-BERT

In [ ]:
ode = MiniBERTNeuralODE(hp=hp).to(device)

## Train

In [ ]:
train(
    model=ode,
    data=data,
    device=device,
    hp=hp,
    model_path="artifacts/ode.pt",
    history_path="artifacts/ode.json"
)

## Eval

In [ ]:
_, _, report, cm = evaluate(ode, data, device)
print(report)

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=labels,
    yticklabels=labels,
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()